1. Difference between "Love" and "love":
"Love" and "love" are treated as different tokens in NLP if case normalization is not applied. Lowercasing ensures both are treated the same.

2. If stopwords are not removed:
The dataset becomes noisy, increases dimensionality, and reduces model efficiency.

3. When removing stopwords is harmful:

Sentiment analysis ("not happy" → removing "not" changes meaning)
Question answering ("what is your name")

4. Stemming vs Lemmatization:

Stemming: Cuts words (running → run)
Lemmatization: Uses dictionary (better accuracy)

Task 2:Build Advanced Preprocessing Function

In [1]:
import re
from collections import Counter

# Exception words
EXCEPTION_WORDS = {"no", "not"}

def preprocess_text(text):
    if not text or text.strip() == "":
        return []

    # Lowercase
    text = text.lower()

    # Remove URLs & emails
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove repeated characters (soooo → soo)
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    # Remove special characters / emojis
    text = re.sub(r'[^\w\s]', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenization
    tokens = text.split()

    # Remove short tokens (≤2 except no/not)
    tokens = [
        word for word in tokens
        if len(word) > 2 or word in EXCEPTION_WORDS
    ]

    return tokens

Task 3: Stress Testing


In [2]:
test_sentences = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

results = []

for sentence in test_sentences:
    tokens = preprocess_text(sentence)
    clean_sentence = " ".join(tokens)

    results.append((sentence, tokens, clean_sentence))

    print("Original:", sentence)
    print("Tokens:", tokens)
    print("Cleaned:", clean_sentence)
    print("-" * 50)

Original: Get 100% FREE access now!!!
Tokens: ['get', 'free', 'access', 'now']
Cleaned: get free access now
--------------------------------------------------
Original: I absolutely looooved this product 😍😍
Tokens: ['absolutely', 'loved', 'this', 'product']
Cleaned: absolutely loved this product
--------------------------------------------------
Original: Worst service ever... 0/10
Tokens: ['worst', 'service', 'ever']
Cleaned: worst service ever
--------------------------------------------------
Original: Call me at 9876543210
Tokens: ['call']
Cleaned: call
--------------------------------------------------
Original: This is THE best course!!!
Tokens: ['this', 'the', 'best', 'course']
Cleaned: this the best course
--------------------------------------------------
Original: Visit https://openai.com now!
Tokens: ['visit', 'now']
Cleaned: visit now
--------------------------------------------------
Original: Nooooo this is baaad!!!
Tokens: ['no', 'this', 'bad']
Cleaned: no this bad
-----

Task 4: Token Analytics


In [3]:
for original, tokens, clean_sentence in results:
    if len(tokens) == 0:
        continue

    total_tokens = len(tokens)
    unique_tokens = len(set(tokens))
    avg_length = sum(len(t) for t in tokens) / total_tokens

    print("Sentence:", original)
    print("Total Tokens:", total_tokens)
    print("Unique Tokens:", unique_tokens)
    print("Avg Token Length:", round(avg_length, 2))
    print("-" * 50)

Sentence: Get 100% FREE access now!!!
Total Tokens: 4
Unique Tokens: 4
Avg Token Length: 4.0
--------------------------------------------------
Sentence: I absolutely looooved this product 😍😍
Total Tokens: 4
Unique Tokens: 4
Avg Token Length: 6.5
--------------------------------------------------
Sentence: Worst service ever... 0/10
Total Tokens: 3
Unique Tokens: 3
Avg Token Length: 5.33
--------------------------------------------------
Sentence: Call me at 9876543210
Total Tokens: 1
Unique Tokens: 1
Avg Token Length: 4.0
--------------------------------------------------
Sentence: This is THE best course!!!
Total Tokens: 4
Unique Tokens: 4
Avg Token Length: 4.25
--------------------------------------------------
Sentence: Visit https://openai.com now!
Total Tokens: 2
Unique Tokens: 2
Avg Token Length: 4.0
--------------------------------------------------
Sentence: Nooooo this is baaad!!!
Total Tokens: 3
Unique Tokens: 3
Avg Token Length: 3.0
-----------------------------------------

Task 5: Frequency Analysis

In [4]:
all_tokens = []

for _, tokens, _ in results:
    all_tokens.extend(tokens)

counter = Counter(all_tokens)

print("Top 10 Frequent Words:")
print(counter.most_common(10))

print("\nLeast 5 Frequent Words:")
print(counter.most_common()[-5:])

Top 10 Frequent Words:
[('this', 4), ('now', 3), ('get', 1), ('free', 1), ('access', 1), ('absolutely', 1), ('loved', 1), ('product', 1), ('worst', 1), ('service', 1)]

Least 5 Frequent Words:
[('limited', 1), ('offer', 1), ('not', 1), ('happy', 1), ('with', 1)]


Task 6: Build Full Pipeline


In [5]:
def full_pipeline(text_list):
    all_tokens = []
    clean_sentences = []

    for text in text_list:
        tokens = preprocess_text(text)
        all_tokens.extend(tokens)
        clean_sentences.append(" ".join(tokens))

    return {
        "tokens": all_tokens,
        "clean_sentences": clean_sentences
    }

# Test
output = full_pipeline(test_sentences)
print(output)

{'tokens': ['get', 'free', 'access', 'now', 'absolutely', 'loved', 'this', 'product', 'worst', 'service', 'ever', 'call', 'this', 'the', 'best', 'course', 'visit', 'now', 'no', 'this', 'bad', 'got', 'win', 'now', 'limited', 'offer', 'not', 'happy', 'with', 'this'], 'clean_sentences': ['get free access now', 'absolutely loved this product', 'worst service ever', 'call', 'this the best course', 'visit now', 'no this bad', 'got', 'win now limited offer', 'not happy with this']}


Task 7: Error Handling


In [6]:
edge_cases = ["", "😂😂😂😂", "123456789"]

for case in edge_cases:
    print("Input:", case)
    print("Output:", preprocess_text(case))
    print("-" * 30)

Input: 
Output: []
------------------------------
Input: 😂😂😂😂
Output: []
------------------------------
Input: 123456789
Output: []
------------------------------
